In [0]:
from pyspark.sql import functions as F, Window

CATALOG = "dbw_taxrisk_dev"

# Expert-set parameters (tunable)
W = {"exposure": 0.40, "gap": 0.30, "evidence": 0.20, "recurrence": 0.10}
GAP_CAP, EVIDENCE_CAP, RECURRENCE_CAP = 0.5, 8, 3

f = spark.table(f"{CATALOG}.gold.creator_features")
e = (spark.table(f"{CATALOG}.gold.creator_exposure")
     .select("creator_key", "expected_value_mid", "tax_exposure_low", "tax_exposure_mid",
             "tax_exposure_high", "value_to_declared_income", "rate_used", "rule_is_placeholder"))

df = f.join(e, "creator_key")

w_pct = Window.orderBy("tax_exposure_mid")
df = (df
    .withColumn("c_exposure", F.percent_rank().over(w_pct))
    .withColumn("c_gap", F.least(F.coalesce(F.col("value_to_declared_income"), F.lit(GAP_CAP)) / GAP_CAP, F.lit(1.0)))
    .withColumn("c_evidence", F.least(F.col("n_in_kind") / EVIDENCE_CAP, F.lit(1.0)))
    .withColumn("c_recurrence", F.least(F.col("venues_repeated") / RECURRENCE_CAP, F.lit(1.0)))
    .withColumn("risk_score", F.round(100 * (
          W["exposure"] * F.col("c_exposure") + W["gap"] * F.col("c_gap")
        + W["evidence"] * F.col("c_evidence") + W["recurrence"] * F.col("c_recurrence")), 1))
    .withColumn("reasons", F.concat_ws("; ",
        F.when(F.col("c_exposure") >= 0.9, F.lit("exposición en el 10% superior")),
        F.when(F.col("value_to_declared_income") >= 0.2, F.lit("valor estimado >= 20% del ingreso declarado")),
        F.when(F.col("n_in_kind") >= 3, F.concat(F.col("n_in_kind").cast("string"), F.lit(" posts clasificados como canje"))),
        F.when(F.col("venues_repeated") >= 2, F.lit("repite venues")),
        F.when(~F.col("is_registered_taxpayer"), F.lit("no registrado como contribuyente"))))
    .withColumn("rank", F.row_number().over(Window.orderBy(F.desc("risk_score"), F.desc("tax_exposure_mid")))))

cases = df.select("rank", "creator_key", "risk_score", "c_exposure", "c_gap", "c_evidence", "c_recurrence",
                  "n_posts", "n_in_kind", "n_collab", "n_unclear", "venues_repeated",
                  "expected_value_mid", "tax_exposure_low", "tax_exposure_mid", "tax_exposure_high",
                  "declared_income", "value_to_declared_income", "is_registered_taxpayer",
                  "rate_used", "rule_is_placeholder", "reasons")

(cases.write.mode("overwrite").option("overwriteSchema", "true")
    .saveAsTable(f"{CATALOG}.gold.risk_cases"))

display(spark.table(f"{CATALOG}.gold.risk_cases").orderBy("rank")
        .select("rank", "risk_score", "n_in_kind", "tax_exposure_mid", "value_to_declared_income", "reasons").limit(15))